# Extraction evaluation: the pilot

Spec 0011 on `full214780-r4-2610131822-0`. The pilot's 50 labels (spec 0011.02) are
judged by the rules of spec 0011.03: per claim, `faithful` (against the label) and
`actual_kind` (against the purpose and risk definitions); per label, the claims that were
missed.

1. Draw and store the pilot.
2. **Judges:** Claude and Codex give the verdict on all 50, independently of the human
   and of each other. Then the labels to route to the human are found.
3. **Human review**, blind, on the human sheet (spec 0011.04), opened here in the notebook:
   the 20 `in_human` labels, and the other pilot labels the judges route to the human, in
   one hashed order.
4. **Reveal:** the judges' verdicts next to the human's where they differ (0011.03 Q15). The
   human decides each and notes rulebook gaps, in a file apart from the blind verdicts.

In [1]:
import hashlib
import json
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import get_args

import pandas as pd
from IPython.display import HTML, IFrame, display

from engine.controllers.claude_controller import ClaudeController
from engine.controllers.codex_controller import CodexController
from engine.models.base_model import Check, Prompt
from engine.repositories.dsld.labels import loader as label_sets
from engine.repositories.dsld.runs import loader as runs
from engine.repositories.dsld.runs import paths as run_paths
from engine.repositories.dsld.runs.schema import STRATA, ActualKind, Faithful, Kind
from engine.repositories.prompts import loader as prompts
from engine.services.extraction_eval.sample import build_pilot
from engine.services.extraction_eval.schema import Quota, SheetCase, SheetClaim

# The human sheet is a frontend beside engine/, under web/ at the repo root.
sys.path.insert(0, str(run_paths.REPO_ROOT))
from web.human_sheet import server as human_sheet_server

RUN = run_paths.RUNS_ROOT / "full214780-r4-2610131822-0"
SEED = 20261020

# Starting quotas, spec 0011.02 Quotas table; only pilot and pilot_human are drawn now.
LARGE = {("both", "rare"): 100, ("both", "stitched"): 150, ("both", "verbatim"): 300,
         ("purpose_only", "rare"): 100, ("purpose_only", "stitched"): 150, ("purpose_only", "verbatim"): 150,
         ("risk_only", "rare"): 100, ("risk_only", "stitched"): 150, ("risk_only", "verbatim"): 150,
         ("neither", None): 300}
HUMAN = {s: 30 if s[0] == "neither" else 20 if s[1] == "verbatim" else 10 for s in STRATA}
QUOTAS = {s: Quota(pilot=5, pilot_human=2, large=LARGE[s], human=HUMAN[s]) for s in STRATA}

## Draw and store the pilot

Written once to `RUN/eva/pilot.jsonl` and `RUN/eva/manifests/pilot.json`; the store
refuses to overwrite, so a rerun of this cell reads the stored pilot instead. To
redraw, delete both files by hand.

In [2]:
if not run_paths.eva_manifest_path(RUN, run_paths.PILOT_MANIFEST_NAME).exists():
    build_pilot(RUN, QUOTAS, SEED)
pilot = runs.read_pilot(RUN)
pilot_ids = [c.label.dsld_id for c in pilot]
pd.DataFrame(
    {"dsld_id": c.label.dsld_id, "group": c.label.stratum[0], "rarest": c.label.stratum[1],
     "rank": c.label.rank, "in_human": c.label.in_human,
     "purpose": len(c.purpose), "risk": len(c.risk)}
    for c in pilot
).groupby(["group", "rarest"], dropna=False).agg(n=("dsld_id", "size"), human=("in_human", "sum"))

n  human
group        rarest            
both         rare      5      2
             stitched  5      2
             verbatim  5      2
neither      NaN       5      2
purpose_only rare      5      2
             stitched  5      2
             verbatim  5      2
risk_only    rare      5      2
             stitched  5      2
             verbatim  5      2

## Judges

Claude and Codex each give the spec 0011.03 verdict on all 50 pilot labels, blind to each
other and to the human. They run before the human review, because the labels they route
to the human join the sheet. Calls run on the subscriptions (spec 0012), never an API key:
Claude through Claude Code's login, Codex through the Responses API on the ChatGPT plan.

The system prompt is the rulebook (`prompts/extraction_judge/rulebook.md`) followed by the
answer format below. `BATCH[judge]` labels go in each call, in `sha256(case_id)` order: 1 for
both judges (0011.03 Q19), so the two judges see the same thing per call. The controller
runs the answer's checks and asks again, with the rule that was broken, when one fails.
Verdicts are appended per label to `RUN/eva/judge_pilot_<judge>.jsonl` together with the
prompt and rulebook hashes. A rerun judges only the labels that have no verdict under the
current prompt, so a change to the rulebook or the answer format re-judges all 50, as
0011.03 asks. A change of model or batch size does not: move the judge's file aside first.

**Blind:** do not open the judge files until your review on the sheet is saved.

The Claude and Codex extractions an earlier version of this notebook made
(`full214780-r4-2610261434-0`, `-1435-0`) stay on disk. They are not used: under 0011.03 the
judges judge Qwen's extraction instead of extracting their own.

In [3]:
JUDGES = {"claude": ClaudeController, "codex": CodexController}
# Labels per call: 1 for both, so the judges are compared on the same input. Codex at 5 moved
# text between the labels of a call; Claude's batch-5 verdicts are kept beside the run's file.
BATCH = {"claude": 1, "codex": 1}
CONFIDENCE = ("high", "medium", "low")

ANSWER_FORMAT = """## Answer format

### Input

You are given one or more labels, each as:

    Label <case_id>

    Text:

        <the label text>

    Claims:

        <claim_id> P: <text>      (extracted as a purpose)
        <claim_id> R: <text>      (extracted as a risk)

A label with no claims shows `(none)` under `Claims:`; for it, list only what it missed. Judge
each label on its own, from its own text only.

### Output

One JSON object and nothing else, no code fence. One entry per label, in the order given,
with its `case_id` exactly as given. In each, one object per claim in the order given, and one
per missed claim (`[]` when there is none), with the fields in this order:

    {"labels": [
      {"case_id": "label:62375",
       "claims": [
         {"claim_id": ..., "rule": ..., "evidence": ..., "reason": ...,
          "faithful": ..., "actual_kind": ..., "confidence": ...}
       ],
       "missed": [
         {"quote": ..., "kind": ..., "rule": ..., "confidence": ...}
       ]}
    ]}

### Claim fields

- `claim_id`: as given.
- `rule`: the id of the rule the verdict rests on, e.g. `"R4a"`, `"D0"`, `"def.risk"`.
- `evidence`: a quote of the label text the verdict rests on; `""` when there is nothing on
  the label to quote.
- `reason`: one sentence.
- `faithful`: `"yes"`, `"invented"` or `"meaning_changed"`.
- `actual_kind`: `"purpose"`, `"risk"`, `"neither"` or `"both"`; `null` when `faithful` is
  `"invented"`.
- `confidence`: `"high"`, `"medium"` or `"low"`.

`rule` and `reason` may be `""` only for a correct claim: `faithful` `"yes"` and `actual_kind`
the kind it was extracted as. Fill both for every other claim.

### Missed fields

- `quote`: a quote of the label text that states the missed claim.
- `kind`: `"purpose"` or `"risk"`.
- `rule`: `"def.purpose"` or `"def.risk"`.
- `confidence`: `"high"`, `"medium"` or `"low"`.

Split missed claims as R3 asks: one entry per claim where each is its own continuous piece of
the text; where a split would need words the label does not repeat (e.g. "consult your doctor
if you are pregnant nursing or taking medication"), quote the whole piece once.

### Quotes

`evidence` and `quote` are copied from the label's own text character for character, as one
continuous piece of one line. Never reword, shorten inside, join pieces, or quote another
label. For a claim stitched from several pieces of a line, `evidence` is that line, or the
part of it that holds all the pieces, never the claim itself."""

RULEBOOK = prompts.load_rulebook("extraction_judge")
SYSTEM = RULEBOOK + "\n\n" + ANSWER_FORMAT
RULEBOOK_SHA = hashlib.sha256(RULEBOOK.encode()).hexdigest()[:16]
PROMPT_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()[:16]

# All 50 pilot labels as the judges see them: the passage and its claims, blind as on the
# sheet, in sha256(case_id) order. The run's label set must be the one the run saw.
manifest = runs.read_manifest(RUN)
passages_sha, labels = label_sets.load_flattened(manifest["label_set"])
if passages_sha != manifest["passages_sha"]:
    raise ValueError(f"label set {manifest['label_set']!r} is not the one run {RUN.name} was made on")
cases = sorted(
    (SheetCase(c.label.case_id, c.label.dsld_id, labels[c.label.dsld_id].passage,
               [*(SheetClaim(x.claim_id, "purpose", x.text) for x in c.purpose),
                *(SheetClaim(x.claim_id, "risk", x.text) for x in c.risk)])
     for c in pilot),
    key=lambda c: hashlib.sha256(c.case_id.encode()).hexdigest(),
)
in_human = {c.label.case_id: c.label.in_human for c in pilot}

# A prompt's user turn -> its labels: the checks get the prompt, and judge it against these.
BATCH_OF: dict[str, list[SheetCase]] = {}


def answer(text: str) -> dict | None:
    """The judge's JSON answer, or None when it is not one of the asked shape. A code fence is tolerated."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        out = json.loads(text)
    except ValueError:
        return None
    shaped = isinstance(out, dict) and isinstance(out.get("labels"), list) and all(
        isinstance(v, dict) and isinstance(v.get("claims"), list) and isinstance(v.get("missed"), list)
        and all(isinstance(x, dict) for x in v["claims"] + v["missed"])
        for v in out["labels"]
    )
    return out if shaped else None


def judged_labels(prompt: Prompt, generation) -> list[tuple[SheetCase, dict]]:
    """Each of the prompt's labels with its verdict, for the labels the answer gives exactly once."""
    out = answer(generation.text)
    if out is None:
        return []
    given = [v.get("case_id") for v in out["labels"]]
    return [(c, out["labels"][given.index(c.case_id)])
            for c in BATCH_OF[prompt.user] if given.count(c.case_id) == 1]


# The checks of 0011.03, one rule each. Each passes when an earlier one already fails, so a
# failure names only what is wrong.
def is_json(prompt, generation):
    return answer(generation.text) is not None


def labels_once(prompt, generation):
    out = answer(generation.text)
    return out is None or (sorted(str(v.get("case_id")) for v in out["labels"])
                           == sorted(c.case_id for c in BATCH_OF[prompt.user]))


def claims_once(prompt, generation):
    return all(sorted(str(x.get("claim_id")) for x in v["claims"]) == sorted(x.claim_id for x in c.claims)
               for c, v in judged_labels(prompt, generation))


def fields_closed(prompt, generation):
    for _, v in judged_labels(prompt, generation):
        for x in v["claims"]:
            if (x.get("faithful") not in get_args(Faithful)
                    or x.get("confidence") not in CONFIDENCE
                    or not all(isinstance(x.get(f), str) for f in ("rule", "evidence", "reason"))):
                return False
            if x["faithful"] == "invented":
                if x.get("actual_kind") is not None:
                    return False
            elif x.get("actual_kind") not in get_args(ActualKind):
                return False
        for m in v["missed"]:
            if (not isinstance(m.get("quote"), str) or m.get("kind") not in get_args(Kind)
                    or not isinstance(m.get("rule"), str) or m.get("confidence") not in CONFIDENCE):
                return False
    return True


def note_given(prompt, generation):
    for c, v in judged_labels(prompt, generation):
        kinds = {x.claim_id: x.kind for x in c.claims}
        for x in v["claims"]:
            correct = x.get("faithful") == "yes" and x.get("actual_kind") == kinds.get(x.get("claim_id"))
            if not correct and not (str(x.get("rule", "")).strip() and str(x.get("reason", "")).strip()):
                return False
    return True


def evidence_in_passage(prompt, generation):
    return all(not (isinstance(e := x.get("evidence"), str) and e.strip()) or e in c.passage
               for c, v in judged_labels(prompt, generation) for x in v["claims"])


def missed_in_passage(prompt, generation):
    return all(not isinstance(q := m.get("quote"), str) or (q.strip() and q in c.passage)
               for c, v in judged_labels(prompt, generation) for m in v["missed"])


def missed_not_claimed(prompt, generation):
    """No missed quote sits on a claim: compared by place on the label, not by text.

    A label can repeat a phrase on another line, and a short quote can match a claim's words by
    chance, so a quote fails only when every place it appears overlaps a place a claim was taken
    from. A stitched claim, not on the label as one piece, has no place and covers nothing.
    """
    for c, v in judged_labels(prompt, generation):
        # Every place each claim's text appears on the label, as (start, end).
        claimed = []
        for x in c.claims:
            i = c.passage.find(x.text) if x.text else -1
            while i != -1:
                claimed.append((i, i + len(x.text)))
                i = c.passage.find(x.text, i + 1)
        for m in v["missed"]:
            q = m.get("quote")
            if not (isinstance(q, str) and q.strip()):
                continue
            # Every place the quote appears; one not in the passage is missed_in_passage's to fail.
            places = []
            i = c.passage.find(q)
            while i != -1:
                places.append((i, i + len(q)))
                i = c.passage.find(q, i + 1)
            if places and all(any(s < ce and cs < e for cs, ce in claimed) for s, e in places):
                return False
    return True


JUDGE_CHECKS = (
    Check("is_json", is_json,
          'Answer with one JSON object {"labels": [...]} and nothing else; each label has "case_id", '
          'a "claims" list and a "missed" list of objects.'),
    Check("labels_once", labels_once,
          'Give exactly one entry per label given, with its case_id exactly as given, prefix included, e.g. "label:62375".'),
    Check("claims_once", claims_once, "Give every claim of a label exactly once, with its claim_id as given, and no other."),
    Check("fields_closed", fields_closed,
          'Fill every field with one of its values: faithful "yes"/"invented"/"meaning_changed"; actual_kind '
          '"purpose"/"risk"/"neither"/"both", null exactly when invented; confidence '
          '"high"/"medium"/"low"; rule, evidence, reason and quote as strings; missed kind "purpose"/"risk".'),
    Check("note_given", note_given,
          "Give rule and reason for every claim that is not faithful yes and of the kind it was extracted as."),
    Check("evidence_in_passage", evidence_in_passage,
          "Copy evidence from the label's own text exactly, character for character, as one continuous piece of "
          "one line: for a stitched claim, the line it comes from, never the claim reworded. Or leave it empty."),
    Check("missed_in_passage", missed_in_passage,
          "Copy every missed quote from the label's own text exactly, character for character, as one continuous "
          "piece of one line. Never reword a quote to split it: where a split needs words the label does not "
          "repeat, quote the whole piece once."),
    Check("missed_not_claimed", missed_not_claimed,
          "Do not list as missed a quote that overlaps an extracted claim: a second claim inside a claim is "
          "part of that claim, and a claim already extracted is not missed."),
)
print(f"{len(cases)} labels, prompt {PROMPT_SHA}, rulebook {RULEBOOK_SHA}, {len(SYSTEM):,} chars of system prompt")

50 labels, prompt 62191c27275e6965, rulebook aa9b5c22f611daa3, 7,923 chars of system prompt


### Run the judges

Claude and Codex run at the same time, one thread each, each `in_flight` calls at a time
(from `configs/*.toml`); their progress lines interleave. Each takes 1 label per call, so 50
calls each. This spends subscription quota. Codex sends the prompt alone, with no tools
(spec 0012, "Codex on the plan token"), and its `ready()` makes one ~20-token test call; if
that asks you to sign in again, run the sign-in cell in notebook 15. A judge that stops
(quota, sign-in) is reported at the end without stopping the other. A call that still fails
its checks after `max_attempts` is listed at the end and stores nothing, so running the cell
again retries it. To see the prompt one call gets before spending anything, print
`works["claude"][0][1].user` with `RUN_JUDGES = False`.

In [4]:
from concurrent.futures import ThreadPoolExecutor

RUN_JUDGES = True  # True spends quota; False only builds the prompts

# Each judge's work: the labels with no verdict under the current prompt, BATCH[name] per prompt.
# Built for both judges before either runs, so the checks find every prompt's labels in BATCH_OF.
works = {}
for name in JUDGES:
    # Resume: a label with a verdict under the current prompt is not judged again.
    path = run_paths.eva_path(RUN, f"judge_pilot_{name}.jsonl")
    stored = [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []
    done = {r["case_id"] for r in stored if r["prompt_sha"] == PROMPT_SHA}
    todo = [c for c in cases if c.case_id not in done]

    # One prompt per BATCH[name] labels: each label's passage indented under Text:, its claims with their ids.
    work = []
    for batch in (todo[i:i + BATCH[name]] for i in range(0, len(todo), BATCH[name])):
        user = "\n\n".join(
            f"Label {c.case_id}\n\nText:\n\n"
            + "\n".join(f"    {line}" for line in c.passage.splitlines())
            + "\n\nClaims:\n\n"
            + ("\n".join(f"    {x.claim_id} {'P' if x.kind == 'purpose' else 'R'}: {x.text}" for x in c.claims)
               or "    (none)")
            for c in batch
        )
        BATCH_OF[user] = batch
        work.append((tuple(c.case_id for c in batch), Prompt(system=SYSTEM, user=user)))
    works[name] = work
    print(f"{name}: {len(done)} of {len(cases)} labels judged under prompt {PROMPT_SHA}; {len(work)} calls to go")


def run_judge(name: str) -> list[dict]:
    """One judge over its work, appending each label's verdict as its call settles; its failures."""
    failed = []
    controller = JUDGES[name]()
    path = run_paths.eva_path(RUN, f"judge_pilot_{name}.jsonl")
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("a") as fh:
            for key, outcome in controller.stream(
                works[name], checks=JUDGE_CHECKS, parse=answer, total=len(works[name]),
                on_progress=lambda p: print(f"  {name} {p.done}/{p.total} {p.detail}"),
            ):
                g = outcome.generation
                if not outcome.ok:
                    failed.append({"judge": name, "batch": key, "attempts": outcome.attempts,
                                   "failed": outcome.failed, "text": g.text[:500]})
                    continue
                judged_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
                for v in outcome.parsed["labels"]:
                    fh.write(json.dumps({
                        "case_id": v["case_id"], "dsld_id": int(v["case_id"].split(":")[1]),
                        "judge": name, "model": g.model, "effort": controller.model.effort,
                        "rulebook_sha": RULEBOOK_SHA, "prompt_sha": PROMPT_SHA,
                        "batch": list(key), "attempts": outcome.attempts, "judged_at": judged_at,
                        "call_latency_ms": g.latency_ms,
                        "call_tokens": [g.usage.prompt_tokens, g.usage.completion_tokens],
                        "claims": v["claims"], "missed": v["missed"],
                    }, ensure_ascii=False) + "\n")
                fh.flush()
    finally:
        controller.close()
    return failed


# Both judges at once, one thread each: separate subscriptions, separate files, nothing shared.
# A judge that stops (ModelUnavailable) raises here after the other has finished.
failures = []
to_run = [name for name in JUDGES if RUN_JUDGES and works[name]]
with ThreadPoolExecutor(max_workers=max(1, len(to_run))) as pool:
    running = {name: pool.submit(run_judge, name) for name in to_run}
errors = {}
for name, future in running.items():
    try:
        failures += future.result()
    except Exception as exc:
        errors[name] = exc
for name, exc in errors.items():
    print(f"{name} stopped: {exc}")

pd.DataFrame(failures) if failures else print("no failed calls")

claude: 50 of 50 labels judged under prompt 62191c27275e6965; 0 calls to go
codex: 50 of 50 labels judged under prompt 62191c27275e6965; 0 calls to go
no failed calls


### Judge agreement and routing

Claude against Codex on the labels both have judged under the current prompt, by
0011.03's routing rule (Q14). A claim is routed when the two differ on `faithful` or
`actual_kind`, or when either gives it `low` confidence. A missed claim is
routed when only one judge lists it (two quotes that overlap count as the same claim), when
the two give it different kinds, or when either gives it `low` confidence.

The table shows only counts and agreement rates, never verdicts, so it does not break your
blindness on the 20 human labels. Adjudicating the routed cases is a later step: your own
verdict comes first, and only then are the judges' verdicts shown (Q15).

In [5]:
# Each judge's verdicts under the current prompt; a label judged twice keeps its last.
judged = {}
for name in JUDGES:
    path = run_paths.eva_path(RUN, f"judge_pilot_{name}.jsonl")
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []
    judged[name] = {r["case_id"]: r for r in rows if r["prompt_sha"] == PROMPT_SHA}
a_name, b_name = JUDGES
both = [c for c in cases if c.case_id in judged[a_name] and c.case_id in judged[b_name]]

claim_rows, missed_rows = [], []
for c in both:
    a, b = judged[a_name][c.case_id], judged[b_name][c.case_id]

    # Claims: the two compared fields, and low confidence from either.
    va, vb = ({x["claim_id"]: x for x in r["claims"]} for r in (a, b))
    for x in c.claims:
        agree = {f: va[x.claim_id][f] == vb[x.claim_id][f] for f in ("faithful", "actual_kind")}
        low = "low" in (va[x.claim_id]["confidence"], vb[x.claim_id]["confidence"])
        claim_rows.append({"case_id": c.case_id, "in_human": in_human[c.case_id], **agree, "low": low,
                           "routed": not all(agree.values()) or low})

    # Missed: each judge's quote against the other's; one counted once, from the judge listing it.
    for mine, theirs, who in ((a["missed"], b["missed"], a_name), (b["missed"], a["missed"], b_name)):
        for m in mine:
            match = next((t for t in theirs if m["quote"] in t["quote"] or t["quote"] in m["quote"]), None)
            if match is not None and who == b_name:
                continue  # already counted from the first judge's side
            same_kind = match is not None and match["kind"] == m["kind"]
            low = m["confidence"] == "low" or (match is not None and match["confidence"] == "low")
            missed_rows.append({"case_id": c.case_id, "in_human": in_human[c.case_id], "listed_by": "both" if match else who,
                                "same_kind": same_kind, "low": low, "routed": match is None or not same_kind or low})

claims_df, missed_df = pd.DataFrame(claim_rows), pd.DataFrame(missed_rows)

# The labels outside the human 20 with anything routed: they join the human sheet.
ROUTED = sorted({r["case_id"] for r in claim_rows + missed_rows if r["routed"] and not r["in_human"]})

print(f"{len(both)} of {len(cases)} labels judged by both under prompt {PROMPT_SHA}")
if len(claims_df):
    print(f"claims: {len(claims_df)}, routed {claims_df.routed.sum()} "
          f"({claims_df.routed.mean():.0%}); agreement "
          + ", ".join(f"{f} {claims_df[f].mean():.0%}" for f in ("faithful", "actual_kind"))
          + f"; low confidence {claims_df.low.sum()}")
if len(missed_df):
    print(f"missed: {len(missed_df)} distinct, listed by {missed_df.listed_by.value_counts().to_dict()}, "
          f"routed {missed_df.routed.sum()}")
print(f"{len(ROUTED)} labels outside the human 20 join the human sheet")
pd.DataFrame({
    "labels": pd.Series([c.case_id for c in both]).map(in_human).value_counts(),
    "claims routed": claims_df.groupby("in_human").routed.sum() if len(claims_df) else 0,
    "missed routed": missed_df.groupby("in_human").routed.sum() if len(missed_df) else 0,
}).rename_axis("in_human").fillna(0).astype(int)

50 of 50 labels judged by both under prompt 62191c27275e6965
claims: 243, routed 15 (6%); agreement faithful 98%, actual_kind 95%; low confidence 3
missed: 48 distinct, listed by {'codex': 19, 'both': 18, 'claude': 11}, routed 39
14 labels outside the human 20 join the human sheet


,labels,claims routed,missed routed
in_human,,,
False,30,3,26
True,20,12,13


## Human review

The next cell starts the human sheet on `127.0.0.1:8765` in the background and opens it
below. The sheet holds the 20 `in_human` labels and the `ROUTED` labels from the cell above,
in one hashed order. No stratum, type, model, or whether a label was routed is shown. On
each label you give the full verdict: per claim, `faithful` and `actual kind` (`both` for a
line holding a purpose and a risk), with `unsure` ticked where it applies. Add missed claims
by selecting text in the label and clicking `+ missed purpose` / `+ missed risk`. **Save
label** checks the verdict and stores it in `RUN/eva/human_pilot.jsonl`. You can save a
label again: the last save counts. The **rules** button opens the rulebook.

The timer in the header counts your time on the shown label. It stops on **pause**, while
the tab is hidden, and when you switch to another label. Each save stores the time since the
label's previous save, so a label's time is the sum over its saves.

If the frame stays blank, open the link the cell prints. VSCode forwards the port from the
SSH box. Running the cell again restarts the server, and saved labels are kept.

In [6]:
PORT = 8765
if "sheet" in globals():
    sheet.shutdown()
    sheet.server_close()
sheet = human_sheet_server.start(RUN, PORT, also=ROUTED)  # loads the label set once: ~15 s

display(HTML(f'<a href="http://localhost:{PORT}/" target="_blank">open the human sheet in a browser tab</a>'))
IFrame(f"http://localhost:{PORT}/", width="100%", height=900)

OSError: [Errno 98] Address already in use

### Review progress

Which labels on the sheet are saved, and how long each took: the active time of all its
saves added up, paused time left out. The table shows counts only, no verdict content, so
you can run it while the review is still going. It does not say which labels are routed
ones.

In [7]:
from engine.services.extraction_eval.human_sheet import sheet_cases

saved = runs.read_human_verdicts(RUN)

# A label's time is every save's active time added up: each save holds the time since the one before.
path = run_paths.eva_path(RUN, run_paths.HUMAN_PILOT_NAME)
seconds = {}
for line in (path.read_text().splitlines() if path.exists() else []):
    try:
        r = json.loads(line)
    except ValueError:
        continue  # a torn last line, as the loader skips it
    seconds[r["case_id"]] = seconds.get(r["case_id"], 0) + r["active_seconds"]

progress = pd.DataFrame(
    {"#": n, "case_id": c.case_id, "claims": len(c.claims), "saved": c.case_id in saved,
     "minutes": round(seconds[c.case_id] / 60, 1) if v else None,
     "unsure": sum(x.unsure for x in v.claims) + sum(x.unsure for x in v.missed) if v else None,
     "missed": len(v.missed) if v else None}
    for n, c in enumerate(sheet_cases(RUN, also=ROUTED), 1) for v in [saved.get(c.case_id)]
)
print(f"{progress.saved.sum()} of {len(progress)} saved, {progress.minutes.sum():.0f} minutes in all")
progress

34 of 34 saved, 50 minutes in all


,#,case_id,claims,saved,minutes,unsure,missed
0,1,label:45585,3,True,0.8,0,0
1,2,label:45288,12,True,2.2,0,0
2,3,label:319840,8,True,4.4,0,0
3,4,label:314001,7,True,2.0,0,0
4,5,label:3585,7,True,2.9,0,0
5,6,label:298733,11,True,2.4,0,0
6,7,label:181305,6,True,0.9,0,0
7,8,label:54007,8,True,1.4,0,0
8,9,label:12472,10,True,2.4,0,0
9,10,label:81524,4,True,1.5,0,0


## Reveal

**Not blind:** this page shows the judges' verdicts next to yours (0011.03 Q15), so open it only
once every label on the sheet is saved. It needs the cells up to "Judge agreement and routing"
to have run, but not the judges themselves.

The next cell starts the reveal page on `127.0.0.1:8766` and opens it below. Each card is one
item on which you and the judges do not all give the same answer, on the labels you reviewed:
missed claims first, then claims. Click your decision on the card and it is stored at once;
**next undecided** takes you on. Clicking a card or a dotted quote finds it in the label text.
The note box is for what the rulebook leaves open, and **rules** opens the rulebook.

You can stop at any time and change a decision later: the last one counts. `human_pilot.jsonl`
is never changed: it stays the blind record that agreement is measured on. Your decisions go to
`RUN/eva/adjudication_pilot.jsonl`, each with your blind value beside it.

If the frame stays blank, open the link the cell prints. Running the cell again restarts the
server, and your decisions are kept.

In [10]:
from web.reveal import server as reveal_server

REVEAL_PORT = 8766
if "reveal_page" in globals():
    reveal_page.shutdown()
    reveal_page.server_close()
reveal_page = reveal_server.start(RUN, cases, judged, REVEAL_PORT)

display(HTML(f'<a href="http://localhost:{REVEAL_PORT}/" target="_blank">open the reveal in a browser tab</a>'))
IFrame(f"http://localhost:{REVEAL_PORT}/", width="100%", height=900)

### Decisions so far

How many items you have decided, how many changed from your blind verdict, and the notes you
left on what the rulebook leaves open. Run it whenever you like; it reads the stored file.

In [11]:
from engine.services.extraction_eval.reveal import reveal_labels

# Only decisions on items the page shows now: a re-judge can change the items.
shown = {(x.case_id, it.item): it.item_kind for x in reveal_labels(RUN, cases, judged) for it in x.items}
done = [a for key, a in runs.read_adjudications(RUN).items() if key in shown]
for kind in ("missed", "claim"):
    of_kind = [a for a in done if a.item_kind == kind]
    print(f"{kind}: {len(of_kind)} of {sum(k == kind for k in shown.values())} decided, "
          f"{sum(a.final != a.blind for a in of_kind)} changed from your blind verdict")
pd.DataFrame({"case_id": a.case_id, "item": a.item, "blind": a.blind, "final": a.final, "gap": a.gap}
             for a in done if a.gap)

missed: 31 of 50 decided, 22 changed from your blind verdict
claim: 22 of 22 decided, 7 changed from your blind verdict


""
